In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.metrics import confusion_matrix, classification_report

In [2]:
df_original = pd.read_csv("card_transdata.csv")

print(df_original.shape)

(1000000, 8)


In [3]:
df_original.dropna(inplace=True)

In [28]:
df_original.corr().T

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
distance_from_home,1.000000,0.000193,-0.001374,0.143124,-0.000697,-0.001622,-0.001301,0.187571
distance_from_last_transaction,0.000193,1.000000,0.001013,-0.000928,0.002055,-0.000899,0.000141,0.091917
ratio_to_median_purchase_price,-0.001374,0.001013,1.000000,0.001374,0.000587,0.000942,-0.000330,0.462305
repeat_retailer,0.143124,-0.000928,0.001374,1.000000,-0.001345,-0.000417,-0.000532,-0.001357
used_chip,-0.000697,0.002055,0.000587,-0.001345,1.000000,-0.001393,-0.000219,-0.060975
used_pin_number,-0.001622,-0.000899,0.000942,-0.000417,-0.001393,1.000000,-0.000291,-0.100293
online_order,-0.001301,0.000141,-0.000330,-0.000532,-0.000219,-0.000291,1.000000,0.191973
fraud,0.187571,0.091917,0.462305,-0.001357,-0.060975,-0.100293,0.191973,1.000000


In [33]:
df_original.shape

(1000000, 8)

In [29]:
df_original.nunique()

,0
distance_from_home,1000000
distance_from_last_transaction,1000000
ratio_to_median_purchase_price,1000000
repeat_retailer,2
used_chip,2
used_pin_number,2
online_order,2
fraud,2


In [4]:
df_original.shape

(1000000, 8)

In [5]:
continuous_cols = [
    "distance_from_home",
    "distance_from_last_transaction",
    "ratio_to_median_purchase_price"
]

binary_cols = [
    "repeat_retailer",
    "used_chip",
    "used_pin_number",
    "online_order"
]

X = df_original[continuous_cols + binary_cols].copy()
y = df_original["fraud"].copy()

print("X:", X.shape)
print("y:", y.shape)

X: (1000000, 7)
y: (1000000,)


In [6]:
SAMPLE_SIZE = 500_000

rng = np.random.RandomState(42)

sample_indices = rng.choice(
    len(X),
    SAMPLE_SIZE,
    replace=False
)

X_sample = X.iloc[sample_indices].copy()
y_sample = y.iloc[sample_indices].copy()

print(X_sample.shape)
print(y_sample.shape)

(500000, 7)
(500000,)


In [7]:
df_original.nunique()

,0
distance_from_home,1000000
distance_from_last_transaction,1000000
ratio_to_median_purchase_price,1000000
repeat_retailer,2
used_chip,2
used_pin_number,2
online_order,2
fraud,2


In [8]:
df_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 8 columns):
 #   Column                          Non-Null Count    Dtype  
---  ------                          --------------    -----  
 0   distance_from_home              1000000 non-null  float64
 1   distance_from_last_transaction  1000000 non-null  float64
 2   ratio_to_median_purchase_price  1000000 non-null  float64
 3   repeat_retailer                 1000000 non-null  float64
 4   used_chip                       1000000 non-null  float64
 5   used_pin_number                 1000000 non-null  float64
 6   online_order                    1000000 non-null  float64
 7   fraud                           1000000 non-null  float64
dtypes: float64(8)
memory usage: 61.0 MB


In [9]:
df_original.columns

Index(['distance_from_home', 'distance_from_last_transaction',
       'ratio_to_median_purchase_price', 'repeat_retailer', 'used_chip',
       'used_pin_number', 'online_order', 'fraud'],
      dtype='object')

In [6]:
continuous_cols = [
    "distance_from_home",
    "distance_from_last_transaction",
    "ratio_to_median_purchase_price"
]

binary_cols = [
    "repeat_retailer",
    "used_chip",
    "used_pin_number",
    "online_order"
]

feature_cols = continuous_cols + binary_cols

X = df_original[feature_cols].copy()
y = df_original["fraud"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1000000, 7)
y shape: (1000000,)


In [7]:
print(y.value_counts())
print()
print("Fraud rate:", y.mean() * 100, "%")

fraud
0.0    912597
1.0     87403
Name: count, dtype: int64

Fraud rate: 8.7403 %


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

In [9]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nTrain fraud rate:",
      y_train.mean() * 100, "%")

print("Test fraud rate :",
      y_test.mean() * 100, "%")

X_train: (700000, 7)
X_test : (300000, 7)

Train fraud rate: 8.740285714285715 %
Test fraud rate : 8.740333333333334 %


In [14]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(
    n_estimators=200,
    contamination=0.10,
    random_state=42,
    n_jobs=-1
)

iso.fit(X_train)

IsolationForest(contamination=0.1, n_estimators=200, n_jobs=-1, random_state=42)

In [15]:
pred = iso.predict(X_test)

In [16]:
anomaly = (pred == -1).astype(int)

In [17]:
print("Test transactions:", len(X_test))
print("Detected anomalies:", anomaly.sum())
print(
    "Anomaly rate:",
    anomaly.mean() * 100,
    "%"
)

Test transactions: 300000
Detected anomalies: 30116
Anomaly rate: 10.038666666666666 %


In [18]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_test,
        anomaly
    )
)

Confusion Matrix:
[[249447  24332]
 [ 20437   5784]]


In [19]:
print("\nClassification Report:")

print(
    classification_report(
        y_test,
        anomaly,
        target_names=[
            "Normal",
            "Fraud"
        ],
        zero_division=0
    )
)


Classification Report:
              precision    recall  f1-score   support

      Normal       0.92      0.91      0.92    273779
       Fraud       0.19      0.22      0.21     26221

    accuracy                           0.85    300000
   macro avg       0.56      0.57      0.56    300000
weighted avg       0.86      0.85      0.86    300000



In [20]:
precision = precision_score(
    y_test,
    anomaly,
    zero_division=0
)

recall = recall_score(
    y_test,
    anomaly,
    zero_division=0
)

f1 = f1_score(
    y_test,
    anomaly,
    zero_division=0
)

print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

Precision: 0.1920573781378669
Recall: 0.2205865527630525
F1: 0.20533574737738963


In [21]:
fraud_rate = (
    y_test[anomaly == 1].mean() * 100
)

print(
    "Fraud % among Isolation Forest anomalies:",
    fraud_rate,
    "%"
)

Fraud % among Isolation Forest anomalies: 19.20573781378669 %


In [22]:
contamination_values = [
    0.03,
    0.05,
    0.075,
    0.10,
    0.125,
    0.15,
    0.20
]

In [23]:
results = []

for contamination in contamination_values:

    print(
        f"Testing contamination = {contamination}"
    )

    iso = IsolationForest(
        n_estimators=200,
        contamination=contamination,
        random_state=42,
        n_jobs=-1
    )

    iso.fit(X_train)

    pred = iso.predict(X_test)

    anomaly = (pred == -1).astype(int)

    precision = precision_score(
        y_test,
        anomaly,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        anomaly,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        anomaly,
        zero_division=0
    )

    anomaly_rate = anomaly.mean() * 100

    fraud_rate = (
        y_test[anomaly == 1].mean() * 100
    )

    results.append([
        contamination,
        anomaly_rate,
        precision,
        recall,
        f1,
        fraud_rate
    ])

Testing contamination = 0.03


KeyboardInterrupt: 

In [ ]:
if_results = pd.DataFrame(
    results,
    columns=[
        "contamination",
        "anomaly_rate",
        "precision",
        "recall",
        "f1",
        "fraud_rate_among_anomalies"
    ]
)

if_results

In [ ]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nFraud rate - train:", y_train.mean())
print("Fraud rate - test :", y_test.mean())

In [ ]:
!pip install lightgbm

#LIGHT GBM FROM HERE

In [11]:
import lightgbm as lgb

In [12]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

scale_pos_weight = neg / pos

print("Normal:", neg)
print("Fraud :", pos)
print("scale_pos_weight:", scale_pos_weight)

Normal: 638818
Fraud : 61182
scale_pos_weight: 10.441273577195908


In [13]:
model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 61182, number of negative: 638818
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.022733 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 773
[LightGBM] [Info] Number of data points in the train set: 700000, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.087403 -> initscore=-2.345767
[LightGBM] [Info] Start training from score -2.345767


LGBMClassifier(learning_rate=0.05, n_estimators=500, n_jobs=-1, random_state=42,
               scale_pos_weight=np.float64(10.441273577195908))

In [14]:
y_prob = model.predict_proba(X_test)[:, 1]

In [43]:
y_pred = (y_prob >= 0.8).astype(int)

In [44]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)


cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("PR-AUC:", average_precision_score(y_test, y_prob))

Confusion Matrix:
[[273587    192]
 [    92  26129]]

Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00    273779
         1.0       0.99      1.00      0.99     26221

    accuracy                           1.00    300000
   macro avg       1.00      1.00      1.00    300000
weighted avg       1.00      1.00      1.00    300000

Precision: 0.9927054443220242
Recall: 0.9964913618855116
F1: 0.994594800350196
PR-AUC: 0.9998441048368174


In [46]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

# Training predictions
train_prob = model.predict_proba(X_train)[:, 1]
train_pred = (train_prob >= 0.8).astype(int)

# Test predictions
test_prob = model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= 0.8).astype(int)

print("===== TRAIN =====")
print("Precision:", precision_score(y_train, train_pred))
print("Recall   :", recall_score(y_train, train_pred))
print("F1       :", f1_score(y_train, train_pred))
print("PR-AUC   :", average_precision_score(y_train, train_prob))

print("\n===== TEST =====")
print("Precision:", precision_score(y_test, test_pred))
print("Recall   :", recall_score(y_test, test_pred))
print("F1       :", f1_score(y_test, test_pred))
print("PR-AUC   :", average_precision_score(y_test, test_prob))

===== TRAIN =====
Precision: 0.997047308319739
Recall   : 0.9989702853780523
F1       : 0.9980078705442432
PR-AUC   : 0.9999850489933132

===== TEST =====
Precision: 0.9927054443220242
Recall   : 0.9964913618855116
F1       : 0.994594800350196
PR-AUC   : 0.9998441048368174


In [ ]:
importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(importance)

In [ ]:
print(df_original.groupby("fraud")[feature_cols].mean())

In [ ]:
print(
    df_original.groupby("fraud")[
        "ratio_to_median_purchase_price"
    ].describe()
)

In [ ]:
feature_cols = [
    "distance_from_home",
    "distance_from_last_transaction",
    "ratio_to_median_purchase_price",
    "repeat_retailer",
    "used_chip",
    "used_pin_number",
    "online_order"
]

print(feature_cols)
print(df_original.columns)

In [ ]:
X_train.columns

In [17]:
import lightgbm as lgb
import pandas as pd
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

# Feature sets
all_features = [
    "distance_from_home",
    "distance_from_last_transaction",
    "ratio_to_median_purchase_price",
    "repeat_retailer",
    "used_chip",
    "used_pin_number",
    "online_order"
]

top3_features = [
    "distance_from_last_transaction",
    "distance_from_home",
    "ratio_to_median_purchase_price"
]

without_strongest = [
    "distance_from_home",
    "ratio_to_median_purchase_price",
    "repeat_retailer",
    "used_chip",
    "used_pin_number",
    "online_order"
]

binary_features = [
    "repeat_retailer",
    "used_chip",
    "used_pin_number",
    "online_order"
]

In [18]:
def evaluate_features(features, name):

    Xtr = X_train[features]
    Xte = X_test[features]

    neg = (y_train == 0).sum()
    pos = (y_train == 1).sum()
    weight = neg / pos

    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        scale_pos_weight=weight,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    model.fit(Xtr, y_train)

    prob = model.predict_proba(Xte)[:, 1]
    pred = (prob >= 0.5).astype(int)

    return {
        "feature_set": name,
        "num_features": len(features),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "pr_auc": average_precision_score(y_test, prob)
    }

In [19]:
results = []

results.append(
    evaluate_features(all_features, "All 7")
)

results.append(
    evaluate_features(top3_features, "Top 3")
)

results.append(
    evaluate_features(without_strongest, "Without strongest")
)

results.append(
    evaluate_features(binary_features, "Binary only")
)

ablation_results = pd.DataFrame(results)

print(ablation_results)

         feature_set  num_features  precision    recall        f1    pr_auc
0              All 7             7   0.988478  0.997902  0.993168  0.999844
1              Top 3             3   0.538797  0.997559  0.699684  0.694846
2  Without strongest             6   0.980778  0.932077  0.955808  0.957287
3        Binary only             4   0.139337  0.971626  0.243723  0.156861


In [20]:
features = [
    "distance_from_home",
    "distance_from_last_transaction",
    "ratio_to_median_purchase_price",
    "repeat_retailer",
    "used_chip",
    "used_pin_number",
    "online_order"
]

results = []

for removed_feature in features:

    current_features = [
        f for f in features
        if f != removed_feature
    ]

    result = evaluate_features(
        current_features,
        f"Without {removed_feature}"
    )

    results.append(result)

loo_results = pd.DataFrame(results)

print(loo_results[
    ["feature_set", "precision", "recall", "f1", "pr_auc"]
])

                              feature_set  precision    recall        f1  \
0              Without distance_from_home   0.948829  0.787766  0.860828   
1  Without distance_from_last_transaction   0.980778  0.932077  0.955808   
2  Without ratio_to_median_purchase_price   0.141053  0.945425  0.245482   
3                 Without repeat_retailer   0.988477  0.997788  0.993111   
4                       Without used_chip   0.858076  0.997483  0.922542   
5                 Without used_pin_number   0.891342  0.997674  0.941515   
6                    Without online_order   0.678576  0.998131  0.807902   

     pr_auc  
0  0.871037  
1  0.957287  
2  0.441055  
3  0.999833  
4  0.979404  
5  0.915095  
6  0.777435  


In [23]:
ratio = (y_train == 0).sum() / (y_train == 1).sum()
ratio

np.float64(10.441273577195908)

In [24]:
from sklearn.model_selection import StratifiedKFold

In [25]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [26]:
cv_results = []

In [34]:
for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_train, y_train), 1
):

    print(f"\n========== FOLD {fold} ==========")

    X_tr = X_train.iloc[train_idx]
    X_val = X_train.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    # Calculate class weight ONLY from this fold's training data
    neg = (y_tr == 0).sum()
    pos = (y_tr == 1).sum()

    fold_weight = neg / pos

    model_cv = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        scale_pos_weight=fold_weight,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    model_cv.fit(X_tr, y_tr)

    # Validation predictions
    y_val_prob = model_cv.predict_proba(X_val)[:, 1]

    y_val_pred = (y_val_prob >= 0.5).astype(int)

    # Metrics
    precision = precision_score(y_val, y_val_pred)
    recall = recall_score(y_val, y_val_pred)
    f1 = f1_score(y_val, y_val_pred)
    pr_auc = average_precision_score(y_val, y_val_prob)

    cv_results.append({
        "fold": fold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "pr_auc": pr_auc
    })

    print(f"Precision: {precision:.6f}")
    print(f"Recall   : {recall:.6f}")
    print(f"F1       : {f1:.6f}")
    print(f"PR-AUC   : {pr_auc:.6f}")


========== FOLD 1 ==========
Precision: 0.980305
Recall   : 0.996649
F1       : 0.988410
PR-AUC   : 0.999694

========== FOLD 2 ==========
Precision: 0.977947
Recall   : 0.996649
F1       : 0.987210
PR-AUC   : 0.999704

========== FOLD 3 ==========
Precision: 0.982108
Recall   : 0.995914
F1       : 0.988963
PR-AUC   : 0.999779

========== FOLD 4 ==========
Precision: 0.983400
Recall   : 0.997303
F1       : 0.990303
PR-AUC   : 0.999782

========== FOLD 5 ==========
Precision: 0.986730
Recall   : 0.996568
F1       : 0.991625
PR-AUC   : 0.999838


In [35]:
cv_df = pd.DataFrame(cv_results)

print("\n===== 5-FOLD RESULTS =====")
print(cv_df)


===== 5-FOLD RESULTS =====
   fold  precision    recall        f1    pr_auc
0     1   0.980305  0.996649  0.988410  0.999694
1     2   0.977947  0.996649  0.987210  0.999704
2     3   0.982108  0.995914  0.988963  0.999779
3     4   0.983400  0.997303  0.990303  0.999782
4     5   0.986730  0.996568  0.991625  0.999838


In [36]:
cv_summary = pd.DataFrame({
    "mean": cv_df[
        ["precision", "recall", "f1", "pr_auc"]
    ].mean(),

    "std": cv_df[
        ["precision", "recall", "f1", "pr_auc"]
    ].std()
})

print("\n===== CV SUMMARY =====")
print(cv_summary)


===== CV SUMMARY =====
               mean       std
precision  0.982098  0.003301
recall     0.996617  0.000492
f1         0.989302  0.001709
pr_auc     0.999759  0.000060


In [37]:
y_tr_prob = model_cv.predict_proba(X_tr)[:, 1]
y_tr_pred = (y_tr_prob >= 0.5).astype(int)

train_f1 = f1_score(y_tr, y_tr_pred)
val_f1 = f1_score(y_val, y_val_pred)

print(f"Train F1: {train_f1:.6f}")
print(f"Val   F1: {val_f1:.6f}")
print(f"Gap    : {train_f1 - val_f1:.6f}")

Train F1: 0.994959
Val   F1: 0.991625
Gap    : 0.003334


In [39]:
cv_results.append({
    "fold": fold,
    "train_f1": train_f1,
    "val_f1": val_f1,
    "f1_gap": train_f1 - val_f1,
    "precision": precision,
    "recall": recall,
    "pr_auc": pr_auc
})
cv_results

[{'fold': 1,
  'precision': 0.9803054662379421,
  'recall': 0.9966492317750899,
  'f1': 0.9884097908899335,
  'pr_auc': np.float64(0.9996943930294712)},
 {'fold': 2,
  'precision': 0.9779470729751404,
  'recall': 0.9966492317750899,
  'f1': 0.9872095847162633,
  'pr_auc': np.float64(0.9997039178789073)},
 {'fold': 3,
  'precision': 0.9821083172147002,
  'recall': 0.995913697286695,
  'f1': 0.9889628307093005,
  'pr_auc': np.float64(0.9997785117425817)},
 {'fold': 4,
  'precision': 0.9834004834810637,
  'recall': 0.997303260603089,
  'f1': 0.9903030794822899,
  'pr_auc': np.float64(0.9997817042275619)},
 {'fold': 5,
  'precision': 0.9867303179868921,
  'recall': 0.9965677862221133,
  'f1': 0.9916246544153521,
  'pr_auc': np.float64(0.9998377994222484)},
 {'fold': 5,
  'train_f1': 0.9949586323257374,
  'val_f1': 0.9916246544153521,
  'f1_gap': 0.003333977910385366,
  'precision': 0.9867303179868921,
  'recall': 0.9965677862221133,
  'pr_auc': np.float64(0.9998377994222484)},
 {'fold': 5,

In [47]:
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_prob = np.zeros(len(X_train))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):

    X_tr = X_train.iloc[train_idx]
    X_val = X_train.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    scale_pos_weight = (y_tr == 0).sum() / (y_tr == 1).sum()

    fold_model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1
    )

    fold_model.fit(X_tr, y_tr)

    # Probability of fraud
    oof_prob[val_idx] = fold_model.predict_proba(X_val)[:, 1]

print("OOF predictions generated.")
print("Shape:", oof_prob.shape)

OOF predictions generated.
Shape: (700000,)


In [48]:
thresholds = np.arange(0.40, 0.91, 0.01)

results = []

for threshold in thresholds:

    y_pred = (oof_prob >= threshold).astype(int)

    precision = precision_score(y_train, y_pred)
    recall = recall_score(y_train, y_pred)
    f1 = f1_score(y_train, y_pred)

    results.append([
        threshold,
        precision,
        recall,
        f1
    ])

In [49]:
import pandas as pd

threshold_results = pd.DataFrame(
    results,
    columns=["threshold", "precision", "recall", "f1"]
)

threshold_results

,threshold,precision,recall,f1
0,0.40,0.980023,0.997483,0.988676
1,0.41,0.980304,0.997336,0.988746
2,0.42,0.980380,0.997221,0.988729
3,0.43,0.980488,0.997107,0.988728
4,0.44,0.980692,0.997042,0.988799
5,0.45,0.980959,0.996976,0.988903
6,0.46,0.981177,0.996845,0.988949
7,0.47,0.981366,0.996780,0.989013
8,0.48,0.981618,0.996747,0.989125
9,0.49,0.981885,0.996682,0.989228


In [50]:
thresholds = np.arange(0.90, 1.001, 0.01)

results_high = []

for threshold in thresholds:

    y_pred = (oof_prob >= threshold).astype(int)

    precision = precision_score(y_train, y_pred)
    recall = recall_score(y_train, y_pred)
    f1 = f1_score(y_train, y_pred)

    results_high.append([
        threshold,
        precision,
        recall,
        f1
    ])

threshold_results_high = pd.DataFrame(
    results_high,
    columns=["threshold", "precision", "recall", "f1"]
)

threshold_results_high

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


,threshold,precision,recall,f1
0,0.90,0.991989,0.989670,0.990828
1,0.91,0.992472,0.989098,0.990782
2,0.92,0.992826,0.988444,0.990630
3,0.93,0.993180,0.987741,0.990453
4,0.94,0.993763,0.987022,0.990381
5,0.95,0.994296,0.985747,0.990003
6,0.96,0.995337,0.983753,0.989511
7,0.97,0.996842,0.980305,0.988504
8,0.98,0.998262,0.976088,0.987050
9,0.99,0.999496,0.971805,0.985456


In [51]:
print("Predictions between 0.4 and 0.6:",
      ((oof_prob >= 0.4) & (oof_prob <= 0.6)).sum())

print("Predictions between 0.5 and 0.9:",
      ((oof_prob >= 0.5) & (oof_prob <= 0.9)).sum())

print("Predictions >= 0.9:",
      (oof_prob >= 0.9).sum())

Predictions between 0.4 and 0.6: 341
Predictions between 0.5 and 0.9: 1048
Predictions >= 0.9: 61039


In [52]:
y_prob_test = model.predict_proba(X_test)[:, 1]

print("Shape:", y_prob_test.shape)

Shape: (300000,)


In [53]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

thresholds = np.arange(0.40, 0.91, 0.01)

test_results = []

for threshold in thresholds:

    y_pred = (y_prob_test >= threshold).astype(int)

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    test_results.append([
        threshold,
        precision,
        recall,
        f1
    ])

threshold_results_test = pd.DataFrame(
    test_results,
    columns=["threshold", "precision", "recall", "f1"]
)

threshold_results_test

,threshold,precision,recall,f1
0,0.40,0.986540,0.997941,0.992208
1,0.41,0.986764,0.997941,0.992321
2,0.42,0.986950,0.997941,0.992415
3,0.43,0.987173,0.997941,0.992528
4,0.44,0.987248,0.997941,0.992565
5,0.45,0.987322,0.997902,0.992584
6,0.46,0.987657,0.997902,0.992753
7,0.47,0.987881,0.997902,0.992866
8,0.48,0.988105,0.997902,0.992979
9,0.49,0.988291,0.997902,0.993074


In [54]:
best_row = threshold_results_test.loc[
    threshold_results_test["f1"].idxmax()
]

print("Best threshold:", best_row["threshold"])
print("Precision:", best_row["precision"])
print("Recall:", best_row["recall"])
print("F1:", best_row["f1"])

Best threshold: 0.7900000000000004
Precision: 0.9927065527065527
Recall: 0.9966439113687502
F1: 0.9946713355916721


In [55]:
from sklearn.metrics import confusion_matrix

BEST_THRESHOLD = best_row["threshold"]

y_pred_test = (
    y_prob_test >= BEST_THRESHOLD
).astype(int)

cm = confusion_matrix(y_test, y_pred_test)

print("Threshold:", BEST_THRESHOLD)

print("\nConfusion Matrix:")
print(cm)

Threshold: 0.7900000000000004

Confusion Matrix:
[[273587    192]
 [    88  26133]]


In [56]:
from sklearn.metrics import (
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_test,
    zero_division=0
))

print("\nFinal Metrics")
print("-------------------------")
print("Threshold :", BEST_THRESHOLD)
print("Precision :", precision_score(y_test, y_pred_test, zero_division=0))
print("Recall    :", recall_score(y_test, y_pred_test, zero_division=0))
print("F1 Score  :", f1_score(y_test, y_pred_test, zero_division=0))
print("ROC-AUC   :", roc_auc_score(y_test, y_prob_test))
print("PR-AUC    :", average_precision_score(y_test, y_prob_test))


Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00    273779
         1.0       0.99      1.00      0.99     26221

    accuracy                           1.00    300000
   macro avg       1.00      1.00      1.00    300000
weighted avg       1.00      1.00      1.00    300000


Final Metrics
-------------------------
Threshold : 0.7900000000000004
Precision : 0.9927065527065527
Recall    : 0.9966439113687502
F1 Score  : 0.9946713355916721
ROC-AUC   : 0.9999850382221188
PR-AUC    : 0.9998441048368174


In [57]:
df_original.shape

(1000000, 8)

In [58]:
df_original.columns

Index(['distance_from_home', 'distance_from_last_transaction',
       'ratio_to_median_purchase_price', 'repeat_retailer', 'used_chip',
       'used_pin_number', 'online_order', 'fraud'],
      dtype='object')

In [59]:
FEATURES = [
    "distance_from_home",
    "distance_from_last_transaction",
    "ratio_to_median_purchase_price",
    "repeat_retailer",
    "used_chip",
    "used_pin_number",
    "online_order"
]

THRESHOLD = 0.79

print("Features:", FEATURES)
print("Number of features:", len(FEATURES))
print("Threshold:", THRESHOLD)

Features: ['distance_from_home', 'distance_from_last_transaction', 'ratio_to_median_purchase_price', 'repeat_retailer', 'used_chip', 'used_pin_number', 'online_order']
Number of features: 7
Threshold: 0.79


In [60]:
class FraudDetector:

    def __init__(self, model, features, threshold=0.79):

        self.model = model
        self.features = features
        self.threshold = threshold

In [69]:
import numbers

def validate_input(self, data):

    if not isinstance(data, dict):
        raise TypeError("Input must be a dictionary.")

    # Check missing features
    missing = set(self.features) - set(data.keys())

    if missing:
        raise ValueError(
            f"Missing features: {sorted(missing)}"
        )

    # Check extra features
    extra = set(data.keys()) - set(self.features)

    if extra:
        raise ValueError(
            f"Unexpected features: {sorted(extra)}"
        )

    # Check all values are numeric
    for feature in self.features:

        if not isinstance(data[feature], numbers.Real):
            raise TypeError(
                f"{feature} must be numeric."
            )

    # Binary features
    binary_features = [
        "repeat_retailer",
        "used_chip",
        "used_pin_number",
        "online_order"
    ]

    for feature in binary_features:

        if data[feature] not in (0, 1, 0.0, 1.0):
            raise ValueError(
                f"{feature} must be 0.0 or 1.0."
            )

    return True

In [62]:
def predict(self, data):

    # Validate input
    self.validate_input(data)

    # Create DataFrame
    input_df = pd.DataFrame(
        [data],
        columns=self.features
    )

    # Get fraud probability
    probability = self.model.predict_proba(
        input_df
    )[0, 1]

    # Apply threshold
    prediction = int(
        probability >= self.threshold
    )

    # Output
    if prediction == 1:
        label = "FRAUD"
    else:
        label = "NORMAL"

    return {
        "fraud_probability": float(probability),
        "threshold": self.threshold,
        "prediction": prediction,
        "label": label
    }

In [64]:
import pandas as pd


class FraudDetector:

    def __init__(self, model, features, threshold=0.79):

        self.model = model
        self.features = features
        self.threshold = threshold


    def validate_input(self, data):

        # Must be dictionary
        if not isinstance(data, dict):
            raise TypeError(
                "Input must be a dictionary."
            )

        # Missing features
        missing = set(self.features) - set(data.keys())

        if missing:
            raise ValueError(
                f"Missing features: {sorted(missing)}"
            )

        # Extra features
        extra = set(data.keys()) - set(self.features)

        if extra:
            raise ValueError(
                f"Unexpected features: {sorted(extra)}"
            )

        # Numeric validation
        for feature in self.features:

            if not isinstance(data[feature], (int, float)):
                raise TypeError(
                    f"{feature} must be numeric."
                )

        # Binary validation
        binary_features = [
            "repeat_retailer",
            "used_chip",
            "used_pin_number",
            "online_order"
        ]

        for feature in binary_features:

            if data[feature] not in [0, 1]:
                raise ValueError(
                    f"{feature} must be either 0 or 1."
                )

        return True


    def predict(self, data):

        # Validate
        self.validate_input(data)

        # Create DataFrame
        input_df = pd.DataFrame(
            [data],
            columns=self.features
        )

        # Fraud probability
        probability = self.model.predict_proba(
            input_df
        )[0, 1]

        # Apply threshold
        prediction = int(
            probability >= self.threshold
        )

        # Label
        label = (
            "FRAUD"
            if prediction == 1
            else "NORMAL"
        )

        return {
            "fraud_probability": float(probability),
            "threshold": self.threshold,
            "prediction": prediction,
            "label": label
        }

In [70]:
FEATURES = [
    "distance_from_home",
    "distance_from_last_transaction",
    "ratio_to_median_purchase_price",
    "repeat_retailer",
    "used_chip",
    "used_pin_number",
    "online_order"
]

THRESHOLD = 0.79

detector = FraudDetector(
    model=model,
    features=FEATURES,
    threshold=THRESHOLD
)

In [71]:
transaction = {
    "distance_from_home": 25.4,
    "distance_from_last_transaction": 8.2,
    "ratio_to_median_purchase_price": 4.7,
    "repeat_retailer": 1,
    "used_chip": 0,
    "used_pin_number": 0,
    "online_order": 1
}
result = detector.predict(transaction)

print(result)

{'fraud_probability': 0.9999967944952186, 'threshold': 0.79, 'prediction': 1, 'label': 'FRAUD'}


In [72]:
bad_transaction = {
    "distance_from_home": 25.4,
    "distance_from_last_transaction": 8.2,
    "ratio_to_median_purchase_price": 4.7,
    "repeat_retailer": 1,
    "used_chip": 0,
    "used_pin_number": 0
}
detector.predict(bad_transaction)

ValueError: Missing features: ['online_order']

In [73]:
bad_transaction = {
    "distance_from_home": 25.4,
    "distance_from_last_transaction": 8.2,
    "ratio_to_median_purchase_price": 4.7,
    "repeat_retailer": 1,
    "used_chip": 0,
    "used_pin_number": 0,
    "online_order": 1,
    "fraud": 0
}
detector.predict(bad_transaction)

ValueError: Unexpected features: ['fraud']

In [74]:
transaction = {
    "distance_from_home": 25.4,
    "distance_from_last_transaction": 8.2,
    "ratio_to_median_purchase_price": 4.7,
    "repeat_retailer": 1,
    "used_chip": -1,
    "used_pin_number": 0,
    "online_order": 1
}
result = detector.predict(transaction)

print(result)

ValueError: used_chip must be either 0 or 1.

In [75]:
import joblib

joblib.dump(
    detector,
    "fraud_detection.pkl"
)

print("Model saved successfully.")

Model saved successfully.


In [76]:
import joblib

detector_loaded = joblib.load(
    "fraud_detection.pkl"
)

In [77]:
print(detector_loaded.features)
print(detector_loaded.threshold)

['distance_from_home', 'distance_from_last_transaction', 'ratio_to_median_purchase_price', 'repeat_retailer', 'used_chip', 'used_pin_number', 'online_order']
0.79


In [78]:
transaction = {
    "distance_from_home": 25.4,
    "distance_from_last_transaction": 8.2,
    "ratio_to_median_purchase_price": 4.7,
    "repeat_retailer": 1,
    "used_chip": 0,
    "used_pin_number": 0,
    "online_order": 1
}

result = detector_loaded.predict(transaction)

print(result)

{'fraud_probability': 0.9999967944952186, 'threshold': 0.79, 'prediction': 1, 'label': 'FRAUD'}


In [79]:
detector = joblib.load("fraud_detection.pkl")

result = detector.predict(transaction)

In [80]:
result

{'fraud_probability': 0.9999967944952186,
 'threshold': 0.79,
 'prediction': 1,
 'label': 'FRAUD'}